In [ ]:
"""
Author:     Aaron Niecestro
Project:    Deep Learning for Wine Quality (DLWQ)

Created:    November 2, 2026
Last Edit:  November 2, 2026
Progress:   Complete

Analysis:   05 - Epoch Optimization
Method:     25 Epochs

Description:
This script evaluates the impact of limiting training
to 25 epochs using the winning neural network
configuration from previous analyses.

The winning configuration from previous analyses is retained:

- StandardScaler
- Sigmoid Activation
- Adam Optimizer
- Architecture: 64 -> 32 -> 16

Research Question:
How many training epochs are required to achieve
optimal predictive performance without unnecessary
training or overfitting?
"""

==========================================
Imports
==========================================

In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import Input

------------------------------------------
Reproducibility
------------------------------------------

In [ ]:
import os
import random
import tensorflow as tf

In [ ]:
SEED = 42

In [ ]:
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
random.seed(SEED)

In [ ]:
np.random.seed(SEED)

In [ ]:
tf.random.set_seed(SEED)

==========================================
Load Data
==========================================

In [ ]:
file_path = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\deep-learning-rl\00-data\winequality-final.csv"
)

In [ ]:
output_dir = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\deep-learning-rl\06-Epoch_Study\tables_figures"
)

In [ ]:
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
dpwq = pd.read_csv(file_path)

In [ ]:
dpwq.columns = dpwq.columns.str.strip()

In [ ]:
print("----- DATA SHAPE (MASTER) -----")

In [ ]:
print(f"Rows: {len(dpwq):,}")
print(f"Columns: {dpwq.shape[1]}")

==========================================
Features and Target
==========================================

In [ ]:
X = dpwq.drop(
    "quality",
    axis=1
)

In [ ]:
y = dpwq["quality"]

==========================================
Encode Wine Type
==========================================

In [ ]:
X = pd.get_dummies(
    X,
    columns=["type"],
    drop_first=True
)

In [ ]:
X["type_white"] = X["type_white"].astype(int)

==========================================
Train Test Split
==========================================

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y
)

==========================================
StandardScaler Scaling
==========================================

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(
    X_train
)

In [ ]:
X_test_scaled = scaler.transform(
    X_test
)

==========================================
Neural Network Architecture
(64 -> 32 -> 16)
==========================================

In [ ]:
model_25e = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    
    Dense(64, activation="sigmoid"),
    Dense(32, activation="sigmoid"),
    Dense(16, activation="sigmoid"),
    
    Dense(1)
])

In [ ]:
model_25e.compile(
    optimizer="adam",
    loss="mse"
)

==========================================
Train Model
==========================================

In [ ]:
start_time = time.time()

In [ ]:
history_25e = model_25e.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=25,
    verbose=0
)

In [ ]:
training_time = (
    time.time() - start_time
)

==========================================
Export Training History
==========================================

In [ ]:
history_df = pd.DataFrame({
    "loss": history_25e.history["loss"],
    "val_loss": history_25e.history["val_loss"]
})

In [ ]:
history_df.to_csv(
    output_dir / "25_epochs_history.csv",
    index=False
)

==========================================
Predictions
==========================================

In [ ]:
y_pred_25e = model_25e.predict(
    X_test_scaled
).flatten()

==========================================
Evaluation Functions
==========================================

In [ ]:
def adjusted_r2(r2, n, p):

    return (
        1 -
        ((1 - r2) * (n - 1))
        /
        (n - p - 1)
    )

In [ ]:
def evaluate_model(
    y_true,
    y_pred,
    p
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    adj_r2 = adjusted_r2(
        r2,
        len(y_true),
        p
    )

    return mae, rmse, adj_r2

==========================================
Evaluation
==========================================

In [ ]:
mae_25e, rmse_25e, adjr2_25e = evaluate_model(
    y_test,
    y_pred_25e,
    X.shape[1]
)

==========================================
Results Table
==========================================

In [ ]:
results_epochs = pd.DataFrame({

    "Epochs": [25],

    "MAE": [mae_25e],

    "RMSE": [rmse_25e],

    "Adjusted_R2": [adjr2_25e],

    "Training_Time_Seconds": [
        training_time
    ],

    "Final_Training_Loss": [
        history_25e.history["loss"][-1]
    ],

    "Final_Validation_Loss": [
        history_25e.history["val_loss"][-1]
    ]
})

In [ ]:
print("\n----- MODEL 25 EPOCHS RESULTS -----")
print(results_epochs.round(4))

==========================================
Export Results
==========================================

In [ ]:
results_epochs.to_csv(
    output_dir / "25_epochs_results.csv",
    index=False
)

==========================================
Learning Curve
==========================================

In [ ]:
plt.figure(figsize=(10,6))

In [ ]:
plt.plot(
    history_25e.history["loss"],
    label="Training Loss"
)

In [ ]:
plt.plot(
    history_25e.history["val_loss"],
    label="Validation Loss"
)

In [ ]:
plt.title(
    "25 Epoch Learning Curve"
)

In [ ]:
plt.xlabel("Epoch")
plt.ylabel("Loss")

In [ ]:
plt.legend()
plt.grid(True)

In [ ]:
plt.savefig(
    output_dir /
    "25_epochs_learning_curve.png",
    bbox_inches="tight"
)

In [ ]:
plt.show()

==========================================
Actual vs Predicted
==========================================

In [ ]:
plt.figure(figsize=(8,8))

In [ ]:
plt.scatter(
    y_test,
    y_pred_25e,
    alpha=0.5
)

In [ ]:
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

In [ ]:
plt.title(
    "25 Epochs: Actual vs Predicted"
)

In [ ]:
plt.xlabel(
    "Actual Quality"
)

In [ ]:
plt.ylabel(
    "Predicted Quality"
)

In [ ]:
plt.savefig(
    output_dir /
    "25_epochs_actual_vs_predicted.png",
    bbox_inches="tight"
)

In [ ]:
plt.show()

==========================================
Export Summary
==========================================

In [ ]:
with open(
    output_dir /
    "25_epochs_summary.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "Analysis 5: Epoch Optimization\n"
    )

    f.write(
        "Method: 25 Epochs\n\n"
    )

    f.write(
        f"MAE: {mae_25e:.4f}\n"
    )

    f.write(
        f"RMSE: {rmse_25e:.4f}\n"
    )

    f.write(
        f"Adjusted R²: {adjr2_25e:.4f}\n"
    )

    f.write(
        f"Training Time: {training_time:.2f} seconds\n"
    )

==========================================
Final Output
==========================================

In [ ]:
print("\n----- 25 EPOCH RESULTS -----")
print(results_epochs.round(4))

In [ ]:
print("\nAnalysis 5 - 25 Epochs Complete")

In [ ]:
print(f"MAE: {mae_25e:.4f}")
print(f"RMSE: {rmse_25e:.4f}")
print(f"Adjusted R²: {adjr2_25e:.4f}")
print(f"Training Time: {training_time:.2f} seconds")